# **HiddenGems:** Analysing NBA contract value with machine learning.

**Module:** Final Year Project<br>
**Student:** Oisin Webb (K00293591)<br>

## **Profiling**
This notebook explores the teams table loaded into PostgreSQL to find data quality issues before cleaning. The raw tables are left untouched, and cleaned versions are saved separately.

## **Cross-referencing source**
Values that look unusual are checked against: 
- **Basketball Reference**. https://www.basketball-reference.com/ 

In [1]:
# Author: Oisin Webb

import sys
sys.path.append("..")

from sqlalchemy import create_engine
from db_config import DB_URL

import pandas as pd

In [2]:
# Author: Oisin Webb
# Database connection
engine = create_engine(DB_URL)

## Profiling Teams Table

In [3]:
# Author: Oisin Webb
# Reading in the teams table in the .raw schema.
teams_df = pd.read_sql("SELECT * FROM raw.teams", engine)
teams_df.shape

(67, 4)

## What teams_df.shape shows me
- 67 rows in the table.
- 4 columns.

In [4]:
# Author: Oisin Webb
teams_df.head(10)

,franchise_id,franchise_name,current_abbr,created_at
0,1,Atlanta Hawks,ATL,2026-07-29 12:34:29.15167+08
1,2,Boston Celtics,BOS,2026-07-29 12:34:29.15167+08
2,3,Brooklyn Nets,BRK,2026-07-29 12:34:29.15167+08
3,4,Chicago Bulls,CHI,2026-07-29 12:34:29.15167+08
4,5,Charlotte Hornets,CHO,2026-07-29 12:34:29.15167+08
5,6,Cleveland Cavaliers,CLE,2026-07-29 12:34:29.15167+08
6,7,Dallas Mavericks,DAL,2026-07-29 12:34:29.15167+08
7,8,Denver Nuggets,DEN,2026-07-29 12:34:29.15167+08
8,9,Detroit Pistons,DET,2026-07-29 12:34:29.15167+08
9,10,Golden State Warriors,GSW,2026-07-29 12:34:29.15167+08


## What teams_df.head shows me
- The columns use `franchise` instead of team as this includes past teams that are no longer around.
- The `created_at` will be dropped as this is not needed at all.

In [5]:
# Author: Oisin Webb
teams_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 67 entries, 0 to 66
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   franchise_id    67 non-null     int64
 1   franchise_name  67 non-null     str  
 2   current_abbr    67 non-null     str  
 3   created_at      67 non-null     str  
dtypes: int64(1), str(3)
memory usage: 5.3 KB


## What teams_df.info() shows me
- All four columns have 0 null rows.

In [6]:
# Author: Oisin Webb
teams_df.describe()

,franchise_id
count,67.000000
mean,34.000000
std,19.485037
min,1.000000
25%,17.500000
50%,34.000000
75%,50.500000
max,67.000000


## What teams_df.describe() shows me
- Since `franchise_id` is our only numeric column it's the only one to appear here.
- `Min` is 1.
- `Max` is 67 in line with the value we got from the `shape` above.

In [7]:
# Author: Oisin Webb
teams_df.isna().sum()

franchise_id      0
franchise_name    0
current_abbr      0
created_at        0
dtype: int64

## What .isna().sum() shows me
- 0 blanks in every column, which matches info() above.

In [8]:
# Author: Oisin Webb
teams_df.duplicated().sum()

np.int64(0)

## What teams_df.duplicated().sum() shows me
- 0 duplicate rows, which matches every franchise_id being unique.

In [9]:
# Author: Oisin Webb
teams_df.duplicated("franchise_name").sum()

np.int64(2)

## What .duplicated().sum() on `franchise_name` shows me
- There are 2 duplicated `franchise_name` in this dataset. Will investigate below.

In [10]:
# Author: Oisin Webb
teams_df[teams_df.duplicated("franchise_name", keep=False)]

,franchise_id,franchise_name,current_abbr,created_at
7,8,Denver Nuggets,DEN,2026-07-29 12:34:29.15167+08
36,37,Denver Nuggets,DNN,2026-07-29 12:34:29.15167+08
46,47,Washington Capitols,WSC,2026-07-29 12:34:29.15167+08
55,56,Washington Capitols,WSA,2026-07-29 12:34:29.15167+08


## Investigating duplicated `franchise_name`.
- I used keep = False as I wanted to see if the teams had different abbreviations.
- The `Denver Nuggets` with the abbreviation `DEN` are the current NBA team the other `Denver Nuggets` with the abbreviation `DNN` is the old team from the 1949-50 season this will be dropped in cleaning.
- `WSC` was the Washington Capitols in the BAA (1946 to 1951) and `WSA` was a different ABA team from 1969-70. Both are long before 2016 so neither is needed.

In [11]:
# Author: Oisin Webb
teams_df["created_at"].value_counts()

created_at
2026-07-29 12:34:29.15167+08    67
Name: count, dtype: int64

## What .value_counts() on `created_at` shows me
- I was going to use the `created_at` to filter the older teams in cleaning but they all appear to have the exact same value. This entire column will be dropped in cleaning.

In [12]:
# Author: Oisin Webb
teams_df.duplicated("current_abbr").sum()

np.int64(0)

## What .duplicated() on `current_abbr` shows me
- There is no duplicated `current_abbr` in that column.

In [13]:
# Author: Oisin Webb
teams_df["franchise_name"].value_counts()

franchise_name
Denver Nuggets            2
Washington Capitols       2
Atlanta Hawks             1
Boston Celtics            1
Brooklyn Nets             1
                         ..
Oakland Oaks              1
Waterloo Hawks            1
Miami Floridians          1
Carolina Cougars          1
Indianapolis Olympians    1
Name: count, Length: 65, dtype: int64

## What value_counts() on `franchise_name` shows me
- 65 different names for 67 rows. The only names that appear twice are the Denver Nuggets and Washington Capitols checked above.
- Most of the list is old teams that no longer exist, such as the Waterloo Hawks and Indianapolis Olympians.

In [14]:
# Author: Oisin Webb
teams_df["franchise_id"].nunique()

67

## What nunique() on `franchise_id` shows me
- 67 unique values for 67 rows, so every team has its own ID and it can be used to link tables.

In [15]:
# Author: Oisin Webb
teams_df["franchise_id"].isna().sum()

np.int64(0)

## What isna() on `franchise_id` shows me
- 0 blanks, so every team has an ID.

## Summary
| Column | Decision | Reason |
|---|---|---|
| franchise_id | Keep | Unique for every team, used to link tables |
| franchise_name | Keep | Full team name, shown in the app |
| current_abbr | Keep | No duplicates, used to link to the stats table |
| created_at | Drop | Same load date on every row, tells us nothing |

- In cleaning only teams that appear in the 2016 to 2025 stats are kept. This removes the old teams in one step instead of one by one manually.